<a href="https://colab.research.google.com/github/Adhisha11/neo4j_ANLP3/blob/main/NLP_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install neo4j openai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 8.0 MB/s eta 0:00:00


In [ ]:
import os
import json
import re
from getpass import getpass

from neo4j import GraphDatabase
from openai import OpenAI
import gradio as gr

In [ ]:
# -----------------------------
# NEO4J CREDENTIALS
# -----------------------------

NEO4J_URI = getpass("Enter Neo4j URI: ")
NEO4J_USERNAME = getpass("Enter Neo4j Username: ")
NEO4J_PASSWORD = getpass("Enter Neo4j Password: ")

# -----------------------------
# OPENAI API KEY
# -----------------------------

OPENAI_API_KEY = getpass("Enter OpenAI API Key:")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

Enter Neo4j URI: ··········
Enter Neo4j Username: ··········
Enter Neo4j Password: ··········
Enter OpenAI API Key:··········


In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

try:
    driver.verify_connectivity()
    print("✅ Successfully connected to Neo4j!")
except Exception as e:
    print("❌ Connection failed")
    print(e)

✅ Successfully connected to Neo4j!


In [ ]:
def run_cypher(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return [record.data() for record in result]


query = """
MATCH (n)
RETURN labels(n) AS node_type, count(*) AS count
"""

results = run_cypher(query)

for row in results:
    print(row)

{'node_type': ['Movie'], 'count': 38}
{'node_type': ['Person'], 'count': 133}


In [ ]:
query = """
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(*) AS count
"""

results = run_cypher(query)

for row in results:
    print(row)

{'relationship': 'ACTED_IN', 'count': 172}
{'relationship': 'DIRECTED', 'count': 44}
{'relationship': 'PRODUCED', 'count': 15}
{'relationship': 'WROTE', 'count': 10}
{'relationship': 'FOLLOWS', 'count': 3}
{'relationship': 'REVIEWED', 'count': 9}


In [ ]:
GRAPH_SCHEMA = """
Node Labels:

Person
- name
- born

Movie
- title
- released
- tagline

Relationships:

(Person)-[:ACTED_IN {roles}]->(Movie)

(Person)-[:DIRECTED]->(Movie)

(Person)-[:PRODUCED]->(Movie)

(Person)-[:WROTE]->(Movie)
"""

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_NAME = "gpt-5"

In [ ]:
def generate_cypher(question):

    prompt = f"""
You are an expert Neo4j Cypher query generator.

Your task is to convert a user's movie-related question into a Cypher query.

Use ONLY this graph schema:

{GRAPH_SCHEMA}

IMPORTANT RULES:

1. Generate ONLY a Cypher query.
2. Do not include explanations.
3. Do not use Markdown.
4. Use only read operations.
5. NEVER use:
   CREATE
   DELETE
   DETACH
   SET
   MERGE
   DROP
   CALL
   LOAD CSV
6. Add LIMIT 20 if the query could return many results.
7. Use the exact labels, properties, and relationships from the schema.

User Question:
{question}
"""

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt
    )

    cypher = response.output_text.strip()

    # Remove accidental markdown fences
    cypher = cypher.replace("```cypher", "")
    cypher = cypher.replace("```", "")
    cypher = cypher.strip()

    return cypher

In [ ]:
FORBIDDEN_KEYWORDS = [
    "CREATE",
    "DELETE",
    "DETACH",
    "SET",
    "MERGE",
    "DROP",
    "CALL",
    "LOAD CSV",
    "REMOVE"
]


def is_safe_cypher(query):

    query_upper = query.upper()

    for keyword in FORBIDDEN_KEYWORDS:
        if keyword in query_upper:
            return False

    return query_upper.startswith(("MATCH", "OPTIONAL MATCH"))

In [ ]:
def execute_generated_cypher(cypher):

    if not is_safe_cypher(cypher):
        return {
            "success": False,
            "data": [],
            "error": "Unsafe Cypher query detected."
        }

    try:

        results = run_cypher(cypher)

        return {
            "success": True,
            "data": results,
            "error": None
        }

    except Exception as e:

        return {
            "success": False,
            "data": [],
            "error": str(e)
        }

In [ ]:
def generate_answer(question, graph_data):

    context = json.dumps(
        graph_data,
        indent=2,
        default=str
    )

    prompt = f"""
You are a helpful Movie Knowledge Graph Assistant.

Answer the user's question using ONLY the information retrieved
from the Neo4j knowledge graph.

User Question:
{question}

Retrieved Graph Data:
{context}

RULES:

1. Do not invent information.
2. Do not use outside knowledge.
3. If the retrieved data is empty, say that you could not find
   the answer in the movie graph.
4. Give a clear and natural answer.
5. When recommending movies, explain why they are relevant
   based on the graph relationships.
"""

    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt
    )

    return response.output_text.strip()

In [ ]:
import re


# ============================================================
# NEO4J QUERY EXECUTOR
# ============================================================

def run_movie_query(cypher, parameters, question=None):

    try:
        with driver.session() as session:

            result = session.run(
                cypher,
                parameters
            )

            records = list(result)

        # No results
        if not records:
            return "❌ No matching results found in the movie database."

        # Extract first column
        values = []

        for record in records:
            value = record[0]

            if value is not None:
                values.append(str(value))

        # Remove duplicates
        values = list(dict.fromkeys(values))

        if not values:
            return "❌ No results found."

        # ====================================================
        # FORMAT RESPONSE
        # ====================================================

        q = question.lower() if question else ""

        # Actor -> Movies
        if (
            "movie" in q
            or "movies" in q
            or "film" in q
            or "films" in q
        ) and (
            "act" in q
            or "acted" in q
            or "star" in q
        ):

            response = "🎬 Movies:\n\n"

            for movie in values:
                response += f"• {movie}\n"

            return response

        # Movie -> Actors
        if (
            "who" in q
            and (
                "acted" in q
                or "actor" in q
                or "starred" in q
            )
        ):

            response = "🎭 Actors:\n\n"

            for actor in values:
                response += f"• {actor}\n"

            return response

        # Director questions
        if "direct" in q:

            response = "🎥 Result:\n\n"

            for value in values:
                response += f"• {value}\n"

            return response

        # Generic response
        return "\n".join(
            f"• {value}" for value in values
        )

    except Exception as e:

        return f"""
❌ Neo4j query failed.

Error:
{str(e)}
"""


# ============================================================
# MAIN QUESTION ANSWERING FUNCTION
# ============================================================

def ask_movie_graph(question):

    q = question.lower().strip()

    # Remove unnecessary punctuation
    q = q.rstrip("?.!")

    # ========================================================
    # 1. PERSON → MOVIES
    #
    # Example:
    # Which movies did Tom Hanks act in?
    # What movies did Keanu Reeves act in?
    # ========================================================

    actor_patterns = [

        r"which movies did (.+?) act in",

        r"what movies did (.+?) act in",

        r"which films did (.+?) act in",

        r"what films did (.+?) act in",

        r"which movies has (.+?) acted in",

        r"what movies has (.+?) acted in",

        r"movies acted in by (.+)",

        r"movies by (.+)"
    ]

    for pattern in actor_patterns:

        match = re.search(pattern, q)

        if match:

            person = match.group(1).strip()

            # Remove trailing punctuation
            person = person.rstrip("?.!")

            # Convert to normal capitalization
            person = " ".join(
                word.capitalize()
                for word in person.split()
            )

            cypher = """
            MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

            WHERE toLower(p.name) = toLower($person)

            RETURN m.title AS movie

            ORDER BY movie
            """

            return run_movie_query(
                cypher,
                {"person": person},
                question
            )


    # ========================================================
    # 2. MOVIE → ACTORS
    #
    # Example:
    # Who acted in The Matrix?
    # Who starred in The Matrix?
    # ========================================================

    actor_in_movie_patterns = [

        r"who acted in (.+)",

        r"who starred in (.+)",

        r"which actors acted in (.+)",

        r"which actors are in (.+)",

        r"actors in (.+)"
    ]

    for pattern in actor_in_movie_patterns:

        match = re.search(pattern, q)

        if match:

            movie = match.group(1).strip()

            movie = movie.rstrip("?.!")

            cypher = """
            MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

            WHERE toLower(m.title) = toLower($movie)

            RETURN p.name AS actor

            ORDER BY actor
            """

            return run_movie_query(
                cypher,
                {"movie": movie},
                question
            )


    # ========================================================
    # 3. MOVIE → DIRECTOR
    #
    # Example:
    # Who directed The Matrix?
    # ========================================================

    director_patterns = [

        r"who directed (.+)",

        r"who is the director of (.+)",

        r"who was the director of (.+)"
    ]

    for pattern in director_patterns:

        match = re.search(pattern, q)

        if match:

            movie = match.group(1).strip()

            movie = movie.rstrip("?.!")

            cypher = """
            MATCH (p:Person)-[:DIRECTED]->(m:Movie)

            WHERE toLower(m.title) = toLower($movie)

            RETURN p.name AS director

            ORDER BY director
            """

            return run_movie_query(
                cypher,
                {"movie": movie},
                question
            )


    # ========================================================
    # 4. DIRECTOR → MOVIES
    #
    # Example:
    # Which movies did Steven Spielberg direct?
    # ========================================================

    director_movie_patterns = [

        r"which movies did (.+?) direct",

        r"what movies did (.+?) direct",

        r"which films did (.+?) direct",

        r"movies directed by (.+)",

        r"films directed by (.+)"
    ]

    for pattern in director_movie_patterns:

        match = re.search(pattern, q)

        if match:

            person = match.group(1).strip()

            person = person.rstrip("?.!")

            person = " ".join(
                word.capitalize()
                for word in person.split()
            )

            cypher = """
            MATCH (p:Person)-[:DIRECTED]->(m:Movie)

            WHERE toLower(p.name) = toLower($person)

            RETURN m.title AS movie

            ORDER BY movie
            """

            return run_movie_query(
                cypher,
                {"person": person},
                question
            )


    # ========================================================
    # 5. SIMPLE "PERSON MOVIES" QUERY
    #
    # Example:
    # Tom Hanks movies
    # Keanu Reeves movies
    # ========================================================

    match = re.search(
        r"^(.+?)\s+(movies|films)$",
        q
    )

    if match:

        person = match.group(1).strip()

        person = person.rstrip("?.!")

        person = " ".join(
            word.capitalize()
            for word in person.split()
        )

        cypher = """
        MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

        WHERE toLower(p.name) = toLower($person)

        RETURN m.title AS movie

        ORDER BY movie
        """

        return run_movie_query(
            cypher,
            {"person": person},
            question
        )


    # ========================================================
    # 6. FALLBACK
    # ========================================================

    return """
❓ I couldn't understand the question.

Try asking:

🎬 Which movies did Tom Hanks act in?

🎬 What movies did Keanu Reeves act in?

🎭 Who acted in The Matrix?

🎥 Who directed The Matrix?

🎥 Which movies did Lana Wachowski direct?
"""

In [ ]:
with driver.session() as session:

    result = session.run("""
        MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

        WHERE toLower(p.name) = "tom hanks"

        RETURN m.title AS movie

        ORDER BY movie
    """)

    for record in result:
        print(record["movie"])

A League of Their Own
Apollo 13
Cast Away
Charlie Wilson's War
Cloud Atlas
Joe Versus the Volcano
Sleepless in Seattle
That Thing You Do
The Da Vinci Code
The Green Mile
The Polar Express
You've Got Mail


In [ ]:
result = ask_movie_graph(
    "Which movies did Tom Hanks act in?"
)

print(result)

🎬 Movies:

• A League of Their Own
• Apollo 13
• Cast Away
• Charlie Wilson's War
• Cloud Atlas
• Joe Versus the Volcano
• Sleepless in Seattle
• That Thing You Do
• The Da Vinci Code
• The Green Mile
• The Polar Express
• You've Got Mail



In [ ]:
with driver.session() as session:

    result = session.run("""
        MATCH (p:Person)-[r]->(m:Movie)

        RETURN
            p.name AS person,
            type(r) AS relationship,
            m.title AS movie

        LIMIT 20
    """)

    for record in result:
        print(
            record["person"],
            "--",
            record["relationship"],
            "-->",
            record["movie"]
        )

Emil Eifrem -- ACTED_IN --> The Matrix
Joel Silver -- PRODUCED --> The Matrix
Lana Wachowski -- DIRECTED --> The Matrix
Lilly Wachowski -- DIRECTED --> The Matrix
Hugo Weaving -- ACTED_IN --> The Matrix
Laurence Fishburne -- ACTED_IN --> The Matrix
Carrie-Anne Moss -- ACTED_IN --> The Matrix
Keanu Reeves -- ACTED_IN --> The Matrix
Joel Silver -- PRODUCED --> The Matrix Reloaded
Lana Wachowski -- DIRECTED --> The Matrix Reloaded
Lilly Wachowski -- DIRECTED --> The Matrix Reloaded
Hugo Weaving -- ACTED_IN --> The Matrix Reloaded
Laurence Fishburne -- ACTED_IN --> The Matrix Reloaded
Carrie-Anne Moss -- ACTED_IN --> The Matrix Reloaded
Keanu Reeves -- ACTED_IN --> The Matrix Reloaded
Joel Silver -- PRODUCED --> The Matrix Revolutions
Lana Wachowski -- DIRECTED --> The Matrix Revolutions
Lilly Wachowski -- DIRECTED --> The Matrix Revolutions
Hugo Weaving -- ACTED_IN --> The Matrix Revolutions
Laurence Fishburne -- ACTED_IN --> The Matrix Revolutions


In [ ]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 110.5 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import re
from neo4j import GraphDatabase


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="Movie Knowledge Graph",
    page_icon="🎬",
    layout="wide"
)


# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #f7f9fc;
}

.title {
    font-size: 42px;
    font-weight: 700;
    text-align: center;
    margin-bottom: 5px;
}

.subtitle {
    text-align: center;
    color: #666;
    font-size: 18px;
    margin-bottom: 30px;
}

.answer-box {
    background-color: #ffffff;
    padding: 25px;
    border-radius: 15px;
    border: 1px solid #e0e0e0;
    box-shadow: 0px 4px 15px rgba(0,0,0,0.05);
}

.result-item {
    padding: 10px 15px;
    margin: 6px 0;
    background-color: #f1f5f9;
    border-radius: 8px;
    font-size: 16px;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# NEO4J CONFIGURATION
# ============================================================

# IMPORTANT:
# Put your actual Neo4j credentials here.

NEO4J_URI = "bolt://44.203.3.206:7687"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "patter-escapes-itinerary"


# ============================================================
# CONNECT TO NEO4J
# ============================================================

@st.cache_resource
def get_driver():

    try:

        driver = GraphDatabase.driver(
            NEO4J_URI,
            auth=(
                NEO4J_USERNAME,
                NEO4J_PASSWORD
            )
        )

        # Verify connection
        driver.verify_connectivity()

        return driver

    except Exception as e:

        st.error(
            f"❌ Could not connect to Neo4j:\n\n{str(e)}"
        )

        return None


driver = get_driver()


# ============================================================
# RUN NEO4J QUERY
# ============================================================

def run_movie_query(cypher, parameters):

    try:

        with driver.session() as session:

            result = session.run(
                cypher,
                parameters
            )

            records = list(result)

        return records

    except Exception as e:

        st.error(
            f"❌ Neo4j query failed:\n\n{str(e)}"
        )

        return []


# ============================================================
# MOVIE QUESTION ANSWERING
# ============================================================

def ask_movie_graph(question):

    q = question.lower().strip()

    q = q.rstrip("?.!")


    # ========================================================
    # PERSON → MOVIES
    # ========================================================

    actor_patterns = [

        r"which movies did (.+?) act in",

        r"what movies did (.+?) act in",

        r"which films did (.+?) act in",

        r"what films did (.+?) act in",

        r"which movies has (.+?) acted in",

        r"what movies has (.+?) acted in",

        r"movies acted in by (.+)",

        r"movies by (.+)"
    ]


    for pattern in actor_patterns:

        match = re.search(pattern, q)

        if match:

            person = match.group(1).strip()

            person = person.rstrip("?.!")

            person = " ".join(
                word.capitalize()
                for word in person.split()
            )


            cypher = """
            MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

            WHERE toLower(p.name) = toLower($person)

            RETURN m.title AS movie

            ORDER BY movie
            """


            records = run_movie_query(
                cypher,
                {"person": person}
            )


            if not records:

                return {
                    "type": "error",
                    "message":
                    f"No movies found for {person}."
                }


            movies = [
                record["movie"]
                for record in records
            ]


            return {
                "type": "movies",
                "person": person,
                "results": movies
            }


    # ========================================================
    # MOVIE → ACTORS
    # ========================================================

    actor_in_movie_patterns = [

        r"who acted in (.+)",

        r"who starred in (.+)",

        r"which actors acted in (.+)",

        r"which actors are in (.+)",

        r"actors in (.+)"
    ]


    for pattern in actor_in_movie_patterns:

        match = re.search(pattern, q)

        if match:

            movie = match.group(1).strip()

            movie = movie.rstrip("?.!")


            cypher = """
            MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

            WHERE toLower(m.title) = toLower($movie)

            RETURN p.name AS actor

            ORDER BY actor
            """


            records = run_movie_query(
                cypher,
                {"movie": movie}
            )


            if not records:

                return {
                    "type": "error",
                    "message":
                    f"No actors found for {movie}."
                }


            actors = [
                record["actor"]
                for record in records
            ]


            return {
                "type": "actors",
                "movie": movie,
                "results": actors
            }


    # ========================================================
    # MOVIE → DIRECTOR
    # ========================================================

    director_patterns = [

        r"who directed (.+)",

        r"who is the director of (.+)",

        r"who was the director of (.+)"
    ]


    for pattern in director_patterns:

        match = re.search(pattern, q)

        if match:

            movie = match.group(1).strip()

            movie = movie.rstrip("?.!")


            cypher = """
            MATCH (p:Person)-[:DIRECTED]->(m:Movie)

            WHERE toLower(m.title) = toLower($movie)

            RETURN p.name AS director

            ORDER BY director
            """


            records = run_movie_query(
                cypher,
                {"movie": movie}
            )


            if not records:

                return {
                    "type": "error",
                    "message":
                    f"No director found for {movie}."
                }


            directors = [
                record["director"]
                for record in records
            ]


            return {
                "type": "directors",
                "movie": movie,
                "results": directors
            }


    # ========================================================
    # DIRECTOR → MOVIES
    # ========================================================

    director_movie_patterns = [

        r"which movies did (.+?) direct",

        r"what movies did (.+?) direct",

        r"which films did (.+?) direct",

        r"movies directed by (.+)",

        r"films directed by (.+)"
    ]


    for pattern in director_movie_patterns:

        match = re.search(pattern, q)

        if match:

            person = match.group(1).strip()

            person = person.rstrip("?.!")

            person = " ".join(
                word.capitalize()
                for word in person.split()
            )


            cypher = """
            MATCH (p:Person)-[:DIRECTED]->(m:Movie)

            WHERE toLower(p.name) = toLower($person)

            RETURN m.title AS movie

            ORDER BY movie
            """


            records = run_movie_query(
                cypher,
                {"person": person}
            )


            if not records:

                return {
                    "type": "error",
                    "message":
                    f"No movies directed by {person}."
                }


            movies = [
                record["movie"]
                for record in records
            ]


            return {
                "type": "movies",
                "person": person,
                "results": movies
            }


    # ========================================================
    # SIMPLE "PERSON MOVIES"
    # ========================================================

    match = re.search(
        r"^(.+?)\s+(movies|films)$",
        q
    )


    if match:

        person = match.group(1).strip()

        person = person.rstrip("?.!")

        person = " ".join(
            word.capitalize()
            for word in person.split()
        )


        cypher = """
        MATCH (p:Person)-[:ACTED_IN]->(m:Movie)

        WHERE toLower(p.name) = toLower($person)

        RETURN m.title AS movie

        ORDER BY movie
        """


        records = run_movie_query(
            cypher,
            {"person": person}
        )


        if not records:

            return {
                "type": "error",
                "message":
                f"No movies found for {person}."
            }


        movies = [
            record["movie"]
            for record in records
        ]


        return {
            "type": "movies",
            "person": person,
            "results": movies
        }


    # ========================================================
    # FALLBACK
    # ========================================================

    return {
        "type": "help",
        "message": "I couldn't understand that question."
    }


# ============================================================
# HEADER
# ============================================================

st.markdown(
    '<div class="title">🎬 Movie Knowledge Graph</div>',
    unsafe_allow_html=True
)

st.markdown(
    '<div class="subtitle">'
    'Ask questions about actors, movies and directors'
    '</div>',
    unsafe_allow_html=True
)


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.header("🎬 Movie Assistant")

    st.write(
        "This chatbot uses a Neo4j knowledge graph "
        "to answer movie-related questions."
    )

    st.divider()

    st.subheader("Try these questions")

    examples = [
        "Which movies did Tom Hanks act in?",
        "What movies did Keanu Reeves act in?",
        "Who acted in The Matrix?",
        "Who directed The Matrix?",
        "Which movies did Lana Wachowski direct?"
    ]

    for example in examples:

        st.write(f"• {example}")

    st.divider()

    st.caption(
        "Powered by Neo4j + Python + Streamlit"
    )


# ============================================================
# QUESTION INPUT
# ============================================================

question = st.text_input(
    "💬 Ask a movie question",
    placeholder="Example: Which movies did Tom Hanks act in?"
)


# ============================================================
# ASK BUTTON
# ============================================================

if st.button(
    "🔍 Ask Question",
    use_container_width=True
):

    if not question.strip():

        st.warning(
            "Please enter a question."
        )

    elif driver is None:

        st.error(
            "Neo4j connection is not available."
        )

    else:

        with st.spinner(
            "Searching the movie knowledge graph..."
        ):

            result = ask_movie_graph(
                question
            )


        # ====================================================
        # DISPLAY RESULT
        # ====================================================

        st.subheader("📋 Answer")


        if result["type"] == "error":

            st.error(
                result["message"]
            )


        elif result["type"] == "help":

            st.warning(
                result["message"]
            )

            st.info(
                """
                Try:

                • Which movies did Tom Hanks act in?

                • Who acted in The Matrix?

                • Who directed The Matrix?

                • Which movies did Lana Wachowski direct?
                """
            )


        elif result["type"] == "movies":

            st.success(
                f"Movies associated with "
                f"**{result['person']}**"
            )

            for movie in result["results"]:

                st.markdown(
                    f"""
                    <div class="result-item">
                    🎬 {movie}
                    </div>
                    """,
                    unsafe_allow_html=True
                )


        elif result["type"] == "actors":

            st.success(
                f"Actors in **{result['movie']}**"
            )

            for actor in result["results"]:

                st.markdown(
                    f"""
                    <div class="result-item">
                    🎭 {actor}
                    </div>
                    """,
                    unsafe_allow_html=True
                )


        elif result["type"] == "directors":

            st.success(
                f"Director(s) of **{result['movie']}**"
            )

            for director in result["results"]:

                st.markdown(
                    f"""
                    <div class="result-item">
                    🎥 {director}
                    </div>
                    """,
                    unsafe_allow_html=True
                )

Overwriting app.py


In [ ]:
from neo4j import GraphDatabase

test_driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

test_driver.verify_connectivity()

print("✅ Neo4j connection successful!")

test_driver.close()

✅ Neo4j connection successful!


In [ ]:
!pkill -f "streamlit run app.py"

In [ ]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > /content/streamlit.log 2>&1 &

In [ ]:
!cat /content/streamlit.log



2026-08-26 05:04:37.662 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.50.183.71:8501



In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print("🌐 OPEN THIS URL:")
print(public_url)

🌐 OPEN THIS URL:
NgrokTunnel: "https://sip-universe-remnant.ngrok-free.dev" -> "http://localhost:8501"
